# Notebook 02 — SQL Analysis

We reframe the observations from notebook 01 as SQL queries against the transaction 
dataset. The point is not the queries themselves — it's that this is how a fraud 
analyst would actually work with the data in a production environment, where the 
transactions live in a database rather than a CSV file.

We load the dataset into an in-memory SQLite database and answer five questions:

1. What is the class distribution?
2. Do the card testing patterns from notebook 01 (`Amount = 0`, `Amount = 1`) show up cleanly in SQL?
3. Which hours have the highest fraud activity?
4. What are the most common fraud amounts?
5. Do fraud amounts vary by time of day?

SQLite is used because it ships with Python natively, requires no server setup, and 
supports standard SQL that would translate to Postgres or MySQL with minimal changes.

## Setup

We import the CSV into a SQLite database using pandas' `to_sql` method. The database 
lives in memory (`:memory:`) so nothing is written to disk — the same setup a data 
analyst would use for one-off exploration on a local machine.

In [1]:
import pandas as pd
import sqlite3

# Load the raw CSV (same as notebook 01)
df = pd.read_csv("../data/creditcard.csv")

# Remove duplicates as we did in notebook 01
df = df.drop_duplicates().reset_index(drop=True)

# Create an in-memory SQLite database and load the table
conn = sqlite3.connect(":memory:")
df.to_sql("transactions", conn, index=False, if_exists="replace")

print(f"Loaded {len(df):,} rows into the transactions table.")

Loaded 283,726 rows into the transactions table.


Before writing any analytical query, we verify the load worked by counting rows and 
listing columns. This is the SQL equivalent of `df.shape` and `df.columns` — a 
one-second check that saves hours of debugging later if something went wrong during 
import.

In [2]:
sanity_check = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Class) AS n_classes
FROM transactions
"""
pd.read_sql(sanity_check, conn)

,total_rows,n_classes
0,283726,2


## 1. Class distribution

The most basic question about any fraud dataset: how many fraudulent transactions are 
there, and what fraction of the total do they represent? This is the SQL equivalent 
of `df["Class"].value_counts()` from notebook 01, but framed as a query an analyst 
would actually run against a production database.

In [3]:
class_distribution = """
SELECT 
    Class,
    COUNT(*) AS n_transactions,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM transactions), 4) AS pct_of_total
FROM transactions
GROUP BY Class
ORDER BY Class
"""
pd.read_sql(class_distribution, conn)

,Class,n_transactions,pct_of_total
0,0,283253,99.8333
1,1,473,0.1667


Same numbers we saw in notebook 01, as expected — 473 fraud cases out of 283,726 total, 
or 0.1667%. The class column and the count agree; SQL and pandas are looking at the 
same data.

## 2. Card testing patterns in Amount

In notebook 01 we found that two amount values dominate the fraud data: `Amount = 0` 
(auth-only requests) and `Amount = 1 EUR` (small-value probes). We now express that 
same finding as a SQL query — the kind an analyst could schedule to run daily against 
production data to flag suspicious volume.

The query bucketizes transactions into three categories (`Amount = 0`, `Amount = 1`, 
everything else) and shows fraud rate within each bucket.

In [4]:
card_testing_patterns = """
SELECT 
    CASE 
        WHEN Amount = 0 THEN 'Amount = 0'
        WHEN Amount = 1 THEN 'Amount = 1'
        ELSE 'Other'
    END AS amount_bucket,
    COUNT(*) AS n_transactions,
    SUM(Class) AS n_fraud,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct
FROM transactions
GROUP BY amount_bucket
ORDER BY fraud_rate_pct DESC
"""
pd.read_sql(card_testing_patterns, conn)

,amount_bucket,n_transactions,n_fraud,fraud_rate_pct
0,Amount = 0,1808,25,1.3827
1,Amount = 1,13566,105,0.7740
2,Other,268352,343,0.1278


The fraud rate is 10.8× higher in `Amount = 0` transactions and 6.0× higher in 
`Amount = 1` transactions compared to any other amount. The findings from notebook 01 
carry over cleanly: card testing shows up as a strong statistical signal that's easy 
to identify with a single SQL query.

An analyst could plug this query into a scheduled job to monitor daily volume in these 
two buckets — a sudden spike in either would be an early indicator of a card testing 
attack in progress.

Note: the ratios here (10.8× and 6.0×) differ from those in notebook 01 (8.4× and 4.7×) 
because they answer slightly different questions. Notebook 01 compares fraud vs legit 
populations ("what fraction of each class has Amount = 0?"). Notebook 02 compares 
amount buckets ("what fraction of transactions in this bucket are fraud?"). The 
underlying counts are identical; only the framing differs.

**Note on ratio differences with notebook 01:** the ratios here (10.8× and 6.0×) 
differ from those in notebook 01 (8.4× and 4.7×) because they answer slightly 
different questions. Notebook 01 compared fraud and legit populations ("what fraction 
of each class has `Amount = 0`?"). Notebook 02 compares amount buckets ("what fraction 
of transactions in this bucket are fraud?"). The underlying counts are identical — 
only the framing differs. Both views are useful: notebook 01's framing describes 
fraud behavior, while notebook 02's framing is closer to how a decision rule would 
actually be applied to an incoming transaction.

## 3. Fraud activity by hour

Notebook 01 showed that fraud is spread thinly across the 48-hour dataset span 
(~10 cases per hour on average), too thin to reveal reliable temporal patterns. 
We re-express that finding in SQL, computing counts and fraud rates per hour.

The query converts `Time` (seconds since first transaction) into an hour bucket 
using integer division, then aggregates by that bucket.

In [5]:
fraud_by_hour = """
SELECT 
    CAST(Time / 3600 AS INTEGER) AS hour_bucket,
    COUNT(*) AS n_transactions,
    SUM(Class) AS n_fraud,
    ROUND(100.0 * SUM(Class) / COUNT(*), 4) AS fraud_rate_pct
FROM transactions
GROUP BY hour_bucket
ORDER BY fraud_rate_pct DESC
LIMIT 10
"""
pd.read_sql(fraud_by_hour, conn)

,hour_bucket,n_transactions,n_fraud,fraud_rate_pct
0,26,1735,27,1.5562
1,28,1122,17,1.5152
2,2,1573,21,1.3350
3,3,1818,13,0.7151
4,7,3366,23,0.6833
5,5,1679,11,0.6552
6,4,1082,6,0.5545
7,11,8474,43,0.5074
8,25,1995,8,0.4010
9,30,2260,6,0.2655


The hours with highest fraud rate are the low-activity hours (26, 28, 2), where 
legit volume is under 1,800 transactions but 17-27 fraud cases give an inflated 
percentage. Meanwhile hour 11 — the single hour with the most fraud cases in absolute 
terms (43) — drops to 8th place here because it's also a high-volume legit hour.

This is the same problem notebook 01 flagged: with only ~10 fraud cases per hour on 
average, small-sample noise dominates any hour-level rate. An operational rule based 
on these numbers would fire on statistical noise, not real patterns. A meaningful 
temporal analysis would require months of data, not 48 hours.


## 4. Most common fraud amounts

In notebook 01 we identified two specific values as strong card testing patterns: 
`Amount = 0` and `Amount = 1`. Here we go further and get the full top 10 fraud 
amounts as a single query, which lets us see whether other specific values also 
concentrate in fraud beyond the two we already flagged.

This kind of query is also useful in production for spotting new specific values as 
they emerge — a merchant-specific test amount that starts appearing in fraud today 
wouldn't be in notebook 01's analysis, but a scheduled version of this query would 
surface it.

In [6]:
top_fraud_amounts = """
SELECT 
    Amount,
    COUNT(*) AS n_fraud
FROM transactions
WHERE Class = 1
GROUP BY Amount
ORDER BY n_fraud DESC
LIMIT 10
"""
pd.read_sql(top_fraud_amounts, conn)

,Amount,n_fraud
0,1.00,105
1,99.99,27
2,0.00,25
3,0.76,17
4,0.77,10
5,0.01,5
6,3.79,4
7,2.00,4
8,12.31,3
9,2.28,3


The top 4 fraud amounts stand out clearly from the rest:

- **€1.00 (105 cases)** and **€0.00 (25 cases)** — the two card testing patterns 
already flagged in notebook 01.
- **€99.99 (27 cases)** and **€0.76-€0.77 combined (27 cases)** — two additional 
non-round values that appear far more often in fraud than any other amount past the 
first two. Both are specific enough to be worth flagging.

Past the top 4, fraud counts drop off — no other amount has more than 5 cases. Those 
tail values may or may not be real patterns, but the sample is too small to say.

## 5. Do fraud amounts vary by time?

The last four queries looked at Amount and Time separately. This one combines them: 
does the average fraud amount change across the 48-hour dataset span, or is it roughly 
constant?

We split the hours into 4 twelve-hour blocks and compare basic amount statistics 
(count, mean, min, max) for fraud within each block.

In [10]:
fraud_amount_stats_by_time = """
SELECT 
    CASE 
        WHEN Time < 6 * 3600 THEN '00-06h'
        WHEN Time < 12 * 3600 THEN '06-12h'
        WHEN Time < 18 * 3600 THEN '12-18h'
        WHEN Time < 24 * 3600 THEN '18-24h'
        WHEN Time < 30 * 3600 THEN '24-30h'
        WHEN Time < 36 * 3600 THEN '30-36h'
        WHEN Time < 42 * 3600 THEN '36-42h'
        ELSE '42-48h'
    END AS time_block,
    COUNT(*) AS n_fraud,
    ROUND(AVG(Amount), 2) AS avg_amount,
    ROUND(MIN(Amount), 2) AS min_amount,
    ROUND(MAX(Amount), 2) AS max_amount
FROM transactions
WHERE Class = 1
GROUP BY time_block
ORDER BY time_block
"""
pd.read_sql(fraud_amount_stats_by_time, conn)

,time_block,n_fraud,avg_amount,min_amount,max_amount
0,00-06h,55,92.45,0.0,1809.68
1,06-12h,91,105.80,0.0,802.52
2,12-18h,71,128.32,0.0,1402.16
3,18-24h,55,157.20,0.0,1354.25
4,24-30h,60,82.78,0.0,829.41
5,30-36h,27,175.71,0.0,2125.87
6,36-42h,62,165.21,0.0,1335.00
7,42-48h,52,118.63,0.0,1504.93


The average fraud amount varies noticeably across the 8 blocks — from €82.78 in 
`24-30h` to €175.71 in `30-36h`. The min-max range shows every block still spans 
from €0 up to €800-2,125, so fraud isn't concentrated in either the low or high end 
depending on time.

The high average in `30-36h` (€175.71) needs a caveat: that block has only 27 fraud 
cases, the smallest sample of any block, so the mean is easily pulled up by one or 
two large outliers. In particular, that block contains the single largest fraudulent 
transaction in the dataset (€2,125.87), which likely drives most of the increase. 
If we set that block aside, the other blocks (52-91 cases each) show averages ranging 
from €82 to €165 — some variation, but not enough to build an operational rule around 
at this dataset size.

## Conclusions

Five SQL queries reframed the fraud patterns identified in notebook 01 as an analyst 
would compute them against a database. Three findings worth carrying forward:

- **The card testing signal is very clean in SQL.** A single `CASE WHEN` query 
identifies `Amount = 0` and `Amount = 1` with fraud rates 10.8× and 6.0× higher than 
the rest of the data. This is exactly the kind of query that could be scheduled to 
run daily against production data to monitor for card testing volume spikes.
- **Two additional specific amounts** (€99.99 and €0.76-€0.77) also concentrate in 
fraud beyond what random chance would explain. They're worth flagging for further 
investigation but not confidently interpretable from this dataset alone.
- **The temporal dimension adds little.** Fraud spans the full amount range at every 
time block, and average tickets vary modestly (€82-€165) across the 48-hour span. 
Small sample sizes per block make hour-level rules unreliable — a meaningful temporal 
analysis would need months of data.

The queries here are all standard SQL and would translate to Postgres, MySQL, or any 
production-grade warehouse with minimal changes.